# Pure Vision Lane Detection — OpenCV Pipeline
**KMUTT FIBO | Machine Vision | Student ID 67340700403**

### Method Overview
```
Front Camera
    │
    ├─► Left lane marking  ─► HSV Yellow Mask → Dilate → Hough Line Fit
    │                                              (weather-adaptive thresholds)
    └─► Right curb edge    ─► Grayscale Canny  → Hough Line Fit
                                                   (weather-adaptive thresholds)
                                          │
                              Ego-Lane Center (normalized x ∈ [0,1])
                                          │
                              Pure Pursuit Controller → CARLA Steer
```

**Detection classes:** `left_marking` (yellow dashed) · `right_edge` (concrete curb)
**Weather conditions:** Clear · Fog · Night · Rain

## 1 · Imports & Setup

In [ ]:
import cv2
import numpy as np
import yaml
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 2 · Region of Interest (ROI)

A **trapezoid polygon** masks the road area visible from the front camera,
discarding sky, buildings, and pavement outside the ego lane.

| Point | x | y | Position |
|-------|---|---|----------|
| 0 | 102 | 892 | bottom-left |
| 1 | 764 | 457 | top-left |
| 2 | 825 | 456 | top-right |
| 3 | 1465 | 894 | bottom-right |

The **split x** (`roi_cx`) is the midpoint of the top edge → separates left and right search areas.

In [ ]:
ROI_YAML = '/home/peeradon/lka-carla-yolo/lka_ws/src/lka_dataset_collection/config/roi.yaml'

def load_roi(path):
    if not __import__('os').path.exists(path):
        return None
    with open(path) as f:
        data = yaml.full_load(f)
    raw = data.get('roi', {}).get('polygon')
    return np.array([list(p) for p in raw], dtype=np.int32) if raw else None

roi_polygon = load_roi(ROI_YAML)

# Split x = midpoint of the two top-edge points (smallest y values)
top_pts = roi_polygon[roi_polygon[:, 1].argsort()][:2]
roi_cx  = int(top_pts[:, 0].mean())

# Build binary mask
sample = cv2.imread('/home/peeradon/lka-carla-yolo/Images/clear.png')
H, W   = sample.shape[:2]
roi_mask = np.zeros((H, W), dtype=np.uint8)
cv2.fillPoly(roi_mask, [roi_polygon], 255)

# Visualise
vis = cv2.cvtColor(cv2.bitwise_and(sample, sample, mask=roi_mask), cv2.COLOR_BGR2RGB)
pts = np.vstack([roi_polygon, roi_polygon[0]])
plt.figure(figsize=(13, 5))
plt.imshow(vis)
plt.plot(pts[:,0], pts[:,1], 'r-', lw=2, label='ROI polygon')
plt.axvline(roi_cx, color='yellow', lw=1.5, ls='--', label=f'split x = {roi_cx}')
plt.title(f'ROI mask  ({W}×{H} px)  |  roi_cx = {roi_cx}')
plt.legend(); plt.axis('off'); plt.tight_layout(); plt.show()
print(f'ROI polygon: {roi_polygon.tolist()}')
print(f'ROI split x (roi_cx): {roi_cx}  |  image centre: {W//2}')

## 3 · Weather-Adaptive Parameters

### Why different thresholds per weather?

| Weather | Challenge | Solution |
|---------|-----------|----------|
| **Clear** | Sun washes out saturation; road surface similar V to marking | Require V ≥ 250 **and** S ≥ 30 |
| **Fog** | Fog nearly eliminates saturation (S ≈ 5) | Very permissive S threshold, rely on H + V |
| **Night** | Street lamps create warm-yellow glow on road (S ≤ 91) | Require S ≥ 150 to separate marking paint (S ≥ 150) |
| **Rain** | Wet road desaturates marking | Medium S threshold |

### Left marking — HSV Yellow mask
| Weather | H | S | V |
|---------|---|---|---|
| clear   | 10–40 | 30–120 | 250–255 |
| fog     | 10–40 |  5–120 | 180–255 |
| night   | 10–40 | 150–255| 30–255  |
| rain    | 15–35 | 25–255 | 150–255 |

### Right curb — Grayscale Canny thresholds
| Weather | low | high |
|---------|-----|------|
| clear   | 30  | 90   |
| fog     | 20  | 60   |
| night   | 20  | 60   |
| rain    | 20  | 60   |

In [ ]:
WEATHER_IMAGES = {
    'clear': '/home/peeradon/lka-carla-yolo/Images/clear.png',
    'fog':   '/home/peeradon/lka-carla-yolo/Images/fog.png',
    'night': '/home/peeradon/lka-carla-yolo/Images/night.png',
    'rain':  '/home/peeradon/lka-carla-yolo/Images/rain.png',
}

# ── Left marking: HSV yellow thresholds ────────────────────────
HSV_LO = {
    'clear': np.array([10,  30, 250]),
    'fog':   np.array([10,   5, 180]),
    'night': np.array([10, 150,  30]),
    'rain':  np.array([15,  25, 150]),
}
HSV_HI = {
    'clear': np.array([40, 120, 255]),
    'fog':   np.array([40, 120, 255]),
    'night': np.array([40, 255, 255]),
    'rain':  np.array([35, 255, 255]),
}

# ── Right curb: grayscale Canny thresholds ─────────────────────
GRAY_CANNY = {
    'clear': (30, 90),
    'fog':   (20, 60),
    'night': (20, 60),
    'rain':  (20, 60),
}

MARGIN = int(W * 0.08)   # dead zone around roi_cx (~128 px)
KERNEL = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
print(f'MARGIN = {MARGIN} px  |  Left search: x < {roi_cx + MARGIN}  |  Right search: x > {roi_cx - MARGIN}')

## 4 · Edge Extraction — All 4 Weather Conditions

Each row shows one weather condition:
- **Col 1** Original image (yellow dashed = split x)
- **Col 2** HSV yellow mask (raw)
- **Col 3** Left edge pixels (dilated mask ∩ ROI, left half only)
- **Col 4** Right edge pixels (gray Canny ∩ ROI, right half only)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(22, 20))
fig.suptitle('Edge Extraction per Weather Condition', fontsize=14, fontweight='bold', y=1.01)

for row, weather in enumerate(['clear', 'fog', 'night', 'rain']):
    img  = cv2.imread(WEATHER_IMAGES[weather])
    hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)

    # Left: HSV yellow → dilate → ROI → left half
    yellow     = cv2.inRange(hsv, HSV_LO[weather], HSV_HI[weather])
    left_edges = cv2.dilate(yellow, KERNEL, iterations=2)
    left_edges = cv2.bitwise_and(left_edges, roi_mask)
    left_edges[:, roi_cx + MARGIN:] = 0

    # Right: gray Canny → ROI → right half
    t1, t2      = GRAY_CANNY[weather]
    right_edges = cv2.Canny(gray, t1, t2)
    right_edges = cv2.bitwise_and(right_edges, roi_mask)
    right_edges[:, :roi_cx - MARGIN] = 0

    titles = [f'{weather} — original', 'HSV yellow mask',
              f'Left edges', f'Right edges (Canny {t1},{t2})']
    imgs   = [cv2.cvtColor(img, cv2.COLOR_BGR2RGB), yellow, left_edges, right_edges]
    cmaps  = [None, 'gray', 'gray', 'gray']

    for col in range(4):
        axes[row, col].imshow(imgs[col], cmap=cmaps[col])
        axes[row, col].set_title(titles[col], fontsize=9)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].axvline(roi_cx, color='yellow', lw=1, ls='--')

plt.tight_layout()
plt.show()

## 5 · Hough Line Fitting

`cv2.HoughLinesP` finds line segments from edge pixels.
A **slope filter** rejects near-horizontal noise and wrong-direction lines.
Three **sanity checks** prevent crossing or out-of-bounds fits.

```
Left  slope: −2.5 ≤ slope ≤ −0.3   (negative = goes up-right toward vanishing point)
Right slope:  0.3 ≤ slope ≤  2.5   (positive = goes up-left toward vanishing point)
```

| Sanity Check | Rule |
|---|---|
| 1. Correct side | Left bottom-x < roi_cx · Right bottom-x > roi_cx |
| 2. No overshoot | Left top-x < roi_cx×1.1 · Right top-x > roi_cx×0.9 |
| 3. No crossing  | left top-x < right top-x |

In [ ]:
def fit_line(points, y_bottom, y_top):
    """Least-squares line fit through (x,y) points. Returns (x_bottom, x_top, coeffs)."""
    if len(points) < 2:
        return None
    pts    = np.array(points)
    coeffs = np.polyfit(pts[:, 1], pts[:, 0], 1)   # x = f(y)
    return int(np.polyval(coeffs, y_bottom)), int(np.polyval(coeffs, y_top)), coeffs


def hough_fit(edge_img, y_bottom, y_top, slope_min, slope_max):
    """Run HoughLinesP, filter by slope, then fit a single line."""
    lines = cv2.HoughLinesP(edge_img, rho=1, theta=np.pi/180,
                             threshold=15, minLineLength=20, maxLineGap=80)
    pts = []
    if lines is not None:
        for seg in lines:
            x1, y1, x2, y2 = seg[0]
            if x2 == x1:
                continue
            slope = (y2 - y1) / (x2 - x1)
            if slope_min <= slope <= slope_max:
                pts.extend([(x1, y1), (x2, y2)])
    return fit_line(pts, y_bottom, y_top)


def apply_sanity_checks(left_fit, right_fit, roi_cx):
    """Reject lines that are on the wrong side, overshoot, or cross."""
    if left_fit  and left_fit[0]  > roi_cx:         left_fit  = None
    if right_fit and right_fit[0] < roi_cx:         right_fit = None
    if left_fit  and left_fit[1]  > roi_cx * 1.1:   left_fit  = None
    if right_fit and right_fit[1] < roi_cx * 0.9:   right_fit = None
    if left_fit and right_fit and left_fit[1] >= right_fit[1]:
        left_fit = None
    return left_fit, right_fit

print('Helper functions defined: fit_line · hough_fit · apply_sanity_checks')

## 6 · Run Pipeline — Select Weather

Change `WEATHER` to test each condition.

In [ ]:
# ── ▼ Change this ▼ ───────────────────────────────────────────
WEATHER = 'clear'   # 'clear' | 'fog' | 'night' | 'rain'
# ──────────────────────────────────────────────────────────────

img  = cv2.imread(WEATHER_IMAGES[WEATHER])
h, w = img.shape[:2]
hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
gray = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)

# Edge extraction
yellow     = cv2.inRange(hsv, HSV_LO[WEATHER], HSV_HI[WEATHER])
left_edges = cv2.dilate(yellow, KERNEL, iterations=2)
left_edges = cv2.bitwise_and(left_edges, roi_mask)
left_edges[:, roi_cx + MARGIN:] = 0

t1, t2      = GRAY_CANNY[WEATHER]
right_edges = cv2.Canny(gray, t1, t2)
right_edges = cv2.bitwise_and(right_edges, roi_mask)
right_edges[:, :roi_cx - MARGIN] = 0

# Hough + sanity checks
y_bottom = h - 1
y_top    = int(h * 0.55)
left_fit  = hough_fit(left_edges,  y_bottom, y_top, slope_min=-2.5, slope_max=-0.3)
right_fit = hough_fit(right_edges, y_bottom, y_top, slope_min= 0.3, slope_max= 2.5)
left_fit, right_fit = apply_sanity_checks(left_fit, right_fit, roi_cx)

# Draw fitted lines
vis = img.copy()
if left_fit:
    cv2.line(vis, (left_fit[0], y_bottom), (left_fit[1], y_top), (0, 255, 0), 3)
    cv2.putText(vis, 'left_marking', (left_fit[0]+10, y_bottom-20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
if right_fit:
    cv2.line(vis, (right_fit[0], y_bottom), (right_fit[1], y_top), (0,255,255), 3)
    cv2.putText(vis, 'right_edge', (right_fit[0]-220, y_bottom-20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

plt.figure(figsize=(14, 5))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axvline(roi_cx, color='yellow', lw=1, ls='--', label=f'roi_cx={roi_cx}')
plt.title(f'Hough Line Fit [{WEATHER}]  —  green: left_marking  |  cyan: right_edge')
plt.legend(fontsize=9); plt.axis('off'); plt.tight_layout(); plt.show()
print(f'[{WEATHER}]  Left={left_fit[0] if left_fit else "MISS"}  Right={right_fit[0] if right_fit else "MISS"}')

## 7 · Ego-Lane Center Calculation

The **reference row** `y_ref = h × 0.85` is used to measure the lane width and center.

```
Both sides detected:   center_x = (lx + rx) / 2
Left only (fallback):  rx = lx + 760 px  →  center_x = lx + 380
Right only (fallback): lx = rx − 760 px  →  center_x = rx − 380
```

**Normalized center** = `center_x / image_width` ∈ [0, 1]  
→ 0.5 means the car is perfectly centred in the lane.  
→ Published on `/lka/lane_center` · `-1` = no detection

In [ ]:
FALLBACK_LANE_PX = 760   # estimated lane width in pixels at y_ref
y_ref = int(h * 0.85)

def compute_center(left_fit, right_fit, y_ref, img_w, fallback_lane_px=760):
    if left_fit is None and right_fit is None:
        return None
    if left_fit and right_fit:
        lx = int(np.polyval(left_fit[2],  y_ref))
        rx = int(np.polyval(right_fit[2], y_ref))
    elif left_fit:
        lx = int(np.polyval(left_fit[2], y_ref))
        rx = lx + fallback_lane_px
    else:
        rx = int(np.polyval(right_fit[2], y_ref))
        lx = rx - fallback_lane_px
    cx           = (lx + rx) // 2
    center_norm  = cx / img_w
    lateral_err  = (img_w / 2) - cx    # pixels; positive = car is left of lane centre
    return cx, center_norm, lateral_err, lx, rx

result = compute_center(left_fit, right_fit, y_ref, w, FALLBACK_LANE_PX)

# ── Final visualisation ────────────────────────────────────────
final = img.copy()

if left_fit and right_fit:
    overlay = final.copy()
    poly_pts = np.array([[left_fit[1], y_top],  [right_fit[1], y_top],
                          [right_fit[0], y_bottom], [left_fit[0], y_bottom]], dtype=np.int32)
    cv2.fillPoly(overlay, [poly_pts], (0, 180, 0))
    final = cv2.addWeighted(overlay, 0.25, final, 0.75, 0)

if left_fit:
    cv2.line(final, (left_fit[0], y_bottom), (left_fit[1], y_top), (0, 255, 0), 3)
    cv2.putText(final, 'left_marking', (left_fit[0]+10, y_bottom-20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
if right_fit:
    cv2.line(final, (right_fit[0], y_bottom), (right_fit[1], y_top), (0,255,255), 3)
    cv2.putText(final, 'right_edge', (right_fit[0]-220, y_bottom-20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

if result:
    cx, cn, err, lx, rx = result
    mode = 'both' if (left_fit and right_fit) else ('left-only' if left_fit else 'right-only')
    cv2.line(final, (lx, y_ref), (rx, y_ref), (180,180,0), 2)
    cv2.circle(final, (cx, y_ref), 14, (0,0,255), -1)
    cv2.line(final, (w//2, y_ref-25), (w//2, y_ref+25), (255,255,0), 2)
    cv2.putText(final, f'[{WEATHER}] center={cn:.3f}  err={err:+.0f}px  ({mode})',
                (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)
    print(f'[{WEATHER}] center_norm={cn:.3f}  lateral_error={err:+.0f}px')
    print(f'  lx={lx}  rx={rx}  lane_width={rx-lx}px  mode={mode}')
else:
    cv2.putText(final, 'NO DETECTION', (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,255), 2)
    print(f'[{WEATHER}] NO DETECTION')

plt.figure(figsize=(16, 7))
plt.imshow(cv2.cvtColor(final, cv2.COLOR_BGR2RGB))
plt.title(f'Final Result [{WEATHER}]  —  red dot: lane centre  |  yellow bar: image centre')
plt.axis('off'); plt.tight_layout(); plt.show()

## 8 · Final Results — All 4 Weather Conditions

Run the full pipeline on all 4 conditions and compare detected lane centres.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(22, 14))
fig.suptitle('Pure Vision Lane Detection — All Weather Conditions', fontsize=14, fontweight='bold')

for ax, weather in zip(axes.flat, ['clear', 'fog', 'night', 'rain']):
    img_w  = cv2.imread(WEATHER_IMAGES[weather])
    h_w, w_w = img_w.shape[:2]
    hsv_w  = cv2.cvtColor(img_w, cv2.COLOR_BGR2HSV)
    gray_w = cv2.GaussianBlur(cv2.cvtColor(img_w, cv2.COLOR_BGR2GRAY), (5, 5), 0)

    yellow_w     = cv2.inRange(hsv_w, HSV_LO[weather], HSV_HI[weather])
    left_e       = cv2.dilate(yellow_w, KERNEL, iterations=2)
    left_e       = cv2.bitwise_and(left_e, roi_mask)
    left_e[:, roi_cx + MARGIN:] = 0

    t1w, t2w     = GRAY_CANNY[weather]
    right_e      = cv2.Canny(gray_w, t1w, t2w)
    right_e      = cv2.bitwise_and(right_e, roi_mask)
    right_e[:, :roi_cx - MARGIN] = 0

    y_bot_w = h_w - 1
    y_top_w = int(h_w * 0.55)
    y_ref_w = int(h_w * 0.85)

    lf = hough_fit(left_e,  y_bot_w, y_top_w, -2.5, -0.3)
    rf = hough_fit(right_e, y_bot_w, y_top_w,  0.3,  2.5)
    lf, rf = apply_sanity_checks(lf, rf, roi_cx)

    res = compute_center(lf, rf, y_ref_w, w_w, FALLBACK_LANE_PX)
    out = img_w.copy()

    if lf and rf:
        ov = out.copy()
        pp = np.array([[lf[1],y_top_w],[rf[1],y_top_w],[rf[0],y_bot_w],[lf[0],y_bot_w]], dtype=np.int32)
        cv2.fillPoly(ov, [pp], (0,180,0))
        out = cv2.addWeighted(ov, 0.25, out, 0.75, 0)

    if lf:
        cv2.line(out, (lf[0], y_bot_w), (lf[1], y_top_w), (0,255,0), 3)
    if rf:
        cv2.line(out, (rf[0], y_bot_w), (rf[1], y_top_w), (0,255,255), 3)

    if res:
        cx_w, cn_w, err_w, lx_w, rx_w = res
        mode_w = 'both' if (lf and rf) else ('L-only' if lf else 'R-only')
        cv2.line(out, (lx_w, y_ref_w), (rx_w, y_ref_w), (180,180,0), 2)
        cv2.circle(out, (cx_w, y_ref_w), 14, (0,0,255), -1)
        cv2.line(out, (w_w//2, y_ref_w-20), (w_w//2, y_ref_w+20), (255,255,0), 2)
        label = f'center={cn_w:.3f}  err={err_w:+.0f}px  [{mode_w}]'
    else:
        label = 'NO DETECTION'
        cv2.putText(out, 'NO DETECTION', (20,50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,255), 2)

    ax.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{weather.upper()}  |  {label}', fontsize=10)
    ax.axis('off')

plt.tight_layout(); plt.show()

## 9 · Parameter Reference

### Hough Transform

| Parameter | Value | Purpose |
|---|---|---|
| `threshold` | 15 | Minimum votes per detected line |
| `minLineLength` | 20 px | Reject very short segments |
| `maxLineGap` | 80 px | Bridge gaps in dashed markings |
| Left slope | −2.5 … −0.3 | Reject horizontal / wrong-direction edges |
| Right slope | 0.3 … 2.5 | Same for right side |

### Geometry

| Variable | Formula | Value |
|---|---|---|
| `y_top` | `h × 0.55` | Top of drawn lane lines |
| `y_ref` | `h × 0.85` | Row used for centre measurement |
| `MARGIN` | `w × 0.08` | Dead zone around `roi_cx` |
| Fallback lane width | — | 760 px |

### ROS2 Integration

| Node | Launch |
|---|---|
| Pure Vision (this pipeline) | `ros2 launch lka_perception pure_vision.launch.py` |
| YOLO detection | `ros2 launch lka_perception lane_detection.launch.py` |
| Pure Pursuit controller | `ros2 launch lka_control lka_controller.launch.py` |

**Output topic:** `/lka/lane_center` (`std_msgs/Float32`)  
- Value ∈ [0, 1] — normalised horizontal position  
- Value = −1 — no detection (controller will brake)